## Initialization

In [1]:
from torch.func import vjp
import torch
from torch.func import jacrev, functional_call
import torch.nn as nn
from torch import Tensor

import torch.nn.functional as F

import sys
import os

current_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root_dir = os.path.abspath(os.path.join(current_notebook_dir, '../../'))

# 将这个父目录添加到sys.path的最前面
if project_root_dir not in sys.path:
    sys.path.insert(0, project_root_dir)

print(sys.path)

['/home/hqdeng7/lijuyang/generalization', '/home/hqdeng7/.conda/envs/ljy/lib/python311.zip', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11/lib-dynload', '', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11/site-packages']


In [2]:
from loss_distribution.pytorch_script.visual_utils \
	import load_cifar10_data, load_model_state_dict

from ntk_result.trials.utils import *

import torchvision
import torchvision.transforms as transforms

data_pth = '/home/hqdeng7/lijuyang/generalization/loss_distribution/pytorch_script/data/cifar10'
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
train_ds = torchvision.datasets.CIFAR10(root=data_pth, train=True, download=True, transform=transform)
test_ds = torchvision.datasets.CIFAR10(root=data_pth, train=False, download=True, transform=transform)

In [3]:
model200_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_200.pth'
model200 = load_model_state_dict('cifar10', 'resnet20', 10, model200_path, 'cuda')
model100_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_100.pth'
model100 = load_model_state_dict('cifar10', 'resnet20', 10, model100_path, 'cuda')

  从字典中提取模型状态字典...
提取成功
  从字典中提取模型状态字典...
提取成功


## Compute all grads

In [4]:
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.func import functional_call, jacrev

# 假设您已经定义了 compute_param_gradients 和 flatten_grads_dict
# ...

# def get_all_gradients(model: nn.Module, ds: Dataset, batch_size: int = 32, device='cuda'):
#     model.to(device)
#     model.eval()

#     dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
#     all_grads = []

#     for inputs, labels in tqdm(dl):
#         grads_per_sample_dict = compute_param_gradients(model, inputs, labels, device=device)
        
#         flattened_grad = flatten_grads_dict(grads_per_sample_dict)
#         all_grads.append(flattened_grad)
#         torch.cuda.empty_cache()

#     all_grads_tensor = torch.cat(all_grads, dim=0)

#     return all_grads_tensor

In [5]:
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.func import functional_call, jacrev

# def compute_all_grads(model: nn.Module, ds: Dataset, device='cuda', batch_size: int = 64):
#     """
#     使用 torch.func.vmap 向量化地获取所有样本的梯度。
#     """
#     model.to(device)
#     model.eval()

#     params = dict(model.named_parameters())
#     buffers = dict(model.named_buffers())

#     dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

#     # 将 compute_single_gradient 定义为内部函数，以便访问 model
#     def compute_single_gradient(p, b, i, l):
#         outputs = functional_call(model, (p, b), i.unsqueeze(0))
#         losses = nn.CrossEntropyLoss(reduction='none')(outputs, l.unsqueeze(0))
#         return losses

#     # vmap 向量化 jacrev，用于批量计算
#     grads_fn = jacrev(compute_single_gradient, argnums=0)
#     vmap_grads_fn = torch.func.vmap(grads_fn, in_dims=(None, None, 0, 0))

#     all_grads = []

#     for inputs, labels in tqdm(dl):
#         inputs, labels = inputs.to(device), labels.to(device)
        
#         # 使用 vmap_grads_fn 进行高效的批量梯度计算
#         grads_per_sample_dict = vmap_grads_fn(params, buffers, inputs, labels)
        
#         flattened_grad = flatten_grads_dict(grads_per_sample_dict)
#         all_grads.append(flattened_grad)
        
#         # 释放缓存以防万一
#         torch.cuda.empty_cache()

#     all_grads_tensor = torch.cat(all_grads, dim=0)

#     return all_grads_tensor

def compute_all_grads(model: nn.Module, ds: Dataset, device='cuda', batch_size: int = 64):
    """
    使用 torch.func.vmap 向量化地获取所有样本的梯度。
    """

    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    all_grads = []

    for inputs, labels in tqdm(dl):
        all_grads.append(compute_param_grads(model100, inputs, labels))
    all_grads_tensor = torch.cat(all_grads, dim=0)

    return all_grads_tensor

In [6]:
all_grads = compute_all_grads(model100, test_ds)

100%|██████████| 157/157 [00:09<00:00, 16.24it/s]


In [7]:
all_grads.requires_grad

False

## high test loss cluster

### cluster in test hloss samples

In [8]:
model100_losses_fn = get_batch_loss_fn(model100)
hloss_samples = {}
hloss_samples[('test', 'epoch100', 'k500')] =find_topk_samples(test_ds, fn=model100_losses_fn, k=500)

100%|██████████| 40/40 [00:02<00:00, 17.01it/s]


In [9]:
hloss_samples_losses = {key: np.array(list(zip(*value))[0]) for key, value in hloss_samples.items()}
hloss_samples_indices = {key: np.array(list(zip(*value))[1]) for key, value in hloss_samples.items()}	

In [10]:
from torch.utils.data import Subset
hloss_sample_grads = {}
hloss_sample_grads[('test', 'epoch100', 'k500')] = \
	compute_all_grads(model100, Subset(test_ds, hloss_samples_indices[('test', 'epoch100', 'k500')]))

100%|██████████| 8/8 [00:00<00:00, 16.83it/s]


In [11]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, KMeans
import matplotlib.pyplot as plt
from sklearn.cluster import Birch
from sklearn.preprocessing import StandardScaler

In [12]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(hloss_sample_grads[('test', 'epoch100', 'k500')].tolist())

In [13]:
def dim_reduc_cluster(X, n_clusters: int, n_components: int = None):
	if n_components != None:
		print("\n使用PCA进行降维...")
		pca = PCA(n_components=0.5, random_state=42)
		X_pca = pca.fit_transform(X)
		print(f"降维后的数据形状: {X_pca.shape}")
	else:
		X_pca = X

	# MiniBatchKMeans比传统的KMeans更适合处理大规模数据，因为它每次只使用一小部分数据进行更新
	print("\n使用MiniBatchKMeans进行聚类...")
	n_clusters = n_clusters
	kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
	kmeans.fit(X_pca)

	# 获取聚类结果
	labels = kmeans.labels_
	print("\n聚类完成！")
	print(f"每个聚类的样本数: {[np.sum(labels == i) for i in range(n_clusters)]}")

	return labels if n_components == None else (X_pca, labels)

In [14]:
X_pca, cls30_labels = dim_reduc_cluster(X_scaled, 30, 500)
cls30_labels


使用PCA进行降维...
降维后的数据形状: (500, 26)

使用MiniBatchKMeans进行聚类...

聚类完成！
每个聚类的样本数: [37, 13, 32, 24, 20, 36, 17, 12, 9, 17, 14, 13, 15, 17, 31, 9, 20, 14, 22, 35, 20, 5, 24, 14, 4, 1, 7, 3, 9, 6]


array([13, 13, 18,  2,  0,  9, 17, 17, 22, 20, 18,  0,  2, 26, 17,  4, 11,
       15,  1, 14, 23, 17,  0,  2,  4, 15,  0,  9,  0, 12, 18, 28,  7,  5,
       25,  6, 23,  0, 10,  5,  9, 14,  3, 11,  4, 19, 22, 12, 16,  9, 18,
        0, 20, 13,  1,  6,  3, 10, 17,  6,  7, 16, 28, 21, 26,  1, 19,  4,
       24, 19, 17, 10, 15,  2,  5, 16, 20,  2,  6, 14,  9, 19,  7, 13, 18,
        2,  5, 18, 13,  5, 12, 23,  4, 23, 20,  9,  5,  2,  2, 20,  5,  5,
        3,  8, 22, 22, 27, 22,  0, 14,  5,  8, 14,  3, 26, 10,  1,  1,  4,
        5,  1, 20,  9, 22,  9, 27,  3,  8, 18, 18, 19,  3, 18,  9, 16,  5,
        7, 22,  0,  0, 16, 19, 16,  5,  2,  3,  0,  2, 16, 23, 18, 18, 12,
        0, 17,  1, 22,  7,  5,  3,  7, 23,  0, 23,  4, 14,  0, 20,  0,  5,
       20, 14, 29,  3, 19, 11, 13, 11,  4, 22,  2,  4, 29,  1, 12, 11, 22,
        0,  2, 15, 16, 18, 19,  5, 28, 22,  5,  0, 19, 14, 10, 22,  8,  2,
       29, 17, 10, 21, 20, 19, 17,  6,  4,  2,  2,  7,  4, 26, 11, 18,  5,
        6, 16, 10,  5, 29

### visualization

In [38]:
def plot_cluster(labels, cluster_idx, len=4):
	fig, axes = plt.subplots(len, len, figsize=(16, 16))
	axes = axes.flatten()
	cluster_sample_indices = hloss_samples_indices[('test', 'epoch100', 'k500')][np.array(labels) == cluster_idx]
	for i in range(len*len):
		img = inverse_trans_cifar10(test_ds[cluster_sample_indices[i]][0])
		img_show(img, ax=axes[i])		


In [ ]:
plot_cluster(cls15_labels, 0, len=5)

In [ ]:
plot_cluster(cls15_labels, 1, len=5)

In [ ]:
plot_cluster(cls15_labels, 2, len=4)

In [ ]:
plot_cluster(cls15_labels, 3, len=4)

In [ ]:
plot_cluster(cls15_labels, 14, len=7)

In [ ]:
plot_cluster(cls15_labels, 13, len=5)

In [57]:

cls30_labels = dim_reduc_cluster(X_pca, 30)


使用MiniBatchKMeans进行聚类...

聚类完成！
每个聚类的样本数: [37, 13, 32, 24, 20, 36, 17, 12, 9, 17, 14, 13, 15, 17, 31, 9, 20, 14, 22, 35, 20, 5, 24, 14, 4, 1, 7, 3, 9, 6]


In [ ]:
plot_cluster(cls30_labels, 0, len=7)

In [ ]:
plot_cluster(cls30_labels, 15, len=5)

In [ ]:
plot_cluster(cls30_labels, 13)

In [ ]:
plot_cluster(cls30_labels, 17, len=3)

In [ ]:
plot_cluster(cls30_labels, 27, len=4)

In [ ]:
plot_cluster(cls30_labels, 28, len=4)

### peek the grad dict

In [ ]:
def compute_param_grads_dict(model: nn.Module, 
						inputs: torch.Tensor,
						labels: torch.Tensor,
						loss_fn = nn.CrossEntropyLoss(reduction='mean'),
						device='cuda'):
	"""
	使用 torch.func.vmap 向量化地获取所有样本的梯度。
	"""
	loss_fn = loss_fn if loss_fn.reduction == 'mean' else type(loss_fn)(reduction='mean')

	model.to(device)
	model.eval()

	params = dict(model.named_parameters())
	buffers = dict(model.named_buffers())

	# 将 compute_single_gradient 定义为内部函数，以便访问 model
	def compute_single_gradient(params, buffers, input, label):
		output = functional_call(model, (params, buffers), input.unsqueeze(0))
		loss = loss_fn(output, label.unsqueeze(0))
		return loss

	# vmap 向量化 jacrev，用于批量计算
	grads_fn = torch.func.grad(compute_single_gradient, argnums=0)
	vmap_grads_fn = torch.func.vmap(grads_fn, in_dims=(None, None, 0, 0))

	inputs, labels = inputs.to(device), labels.to(device)
	
	# 使用 vmap_grads_fn 进行高效的批量梯度计算
	grads_per_sample_dict = vmap_grads_fn(params, buffers, inputs, labels)
	
	# 释放缓存以防万一
	torch.cuda.empty_cache()

	return grads_per_sample_dict

In [ ]:
input, label = test_ds[hloss_samples_indices[('test', 'epoch100', 'k500')][2]]
input_, label_ = input.unsqueeze(0), torch.tensor(label).unsqueeze(0)
compute_param_grads_dict(model100, input_, label_)

In [ ]:
hloss_samples_indices[('test', 'epoch100', 'k500')]

### clusters internal cos sim

In [16]:
cls30_indices_arrs = [hloss_samples_indices[('test', 'epoch100', 'k500')][cls30_labels == i] for i in range(30)]


In [17]:
def compute_single_gradient(model: nn.Module, input, label, loss_fn=nn.CrossEntropyLoss(reduction='mean')):
	input_ = input.unsqueeze(0)
	label_ = torch.tensor(label).unsqueeze(0)
	return compute_param_grads(model, input_, label_, loss_fn).squeeze(0)

In [18]:
min_i = 0
min_j = 0
min_cos_sim = 1
tar_arr = cls30_indices_arrs[0]

for i in tqdm(range(len(tar_arr))):
	for j in range(i+1, len(tar_arr)):
		input1, label1 = test_ds[tar_arr[i]]
		input2, label2 = test_ds[tar_arr[j]]
		grad1 = compute_single_gradient(model100, input1, label1)
		grad2 = compute_single_gradient(model100, input2, label2)
		cos_sim = F.cosine_similarity(grad1, grad2, dim=0)
		if cos_sim < min_cos_sim:
			min_cos_sim = cos_sim
			min_i, min_j = i, j

print(min_i, min_j, min_cos_sim)

100%|██████████| 55/55 [02:03<00:00,  2.24s/it]

12 17 tensor(-0.1314, device='cuda:0')


In [ ]:
input1, label1 = test_ds[tar_arr[min_i]]
input2, label2 = test_ds[cls30_indices_arrs[1][27]]
grad1 = compute_single_gradient(model100, input1, label1)
grad2 = compute_single_gradient(model100, input2, label2)
cos_sim = F.cosine_similarity(grad1, grad2, dim=0)
cos_sim

In [ ]:
min_i = 0
min_j = 0
min_cos_sim = 1
tar_arr = cls30_indices_arrs[15]

for i in tqdm(range(len(tar_arr))):
	for j in range(i+1, len(tar_arr)):
		input1, label1 = test_ds[tar_arr[i]]
		input2, label2 = test_ds[tar_arr[j]]
		grad1 = compute_single_gradient(model100, input1, label1)
		grad2 = compute_single_gradient(model100, input2, label2)
		cos_sim = F.cosine_similarity(grad1, grad2, dim=0)
		if cos_sim < min_cos_sim:
			min_cos_sim = cos_sim
			min_i, min_j = i, j

print(min_i, min_j, min_cos_sim)

## training clusters from test

### preparation

In [58]:
cls30_indices_arrs = [hloss_samples_indices[('test', 'epoch100', 'k500')][cls30_labels == i] for i in range(30)]

### using first one to cluster

In [59]:
tar_idx = cls30_indices_arrs[0][0]
ref_input, ref_label = test_ds[tar_idx]
fn = get_batch_grad_cos_fn(model100, ref_input, ref_label)
top500_train_cos_sim = find_topk_samples(train_ds, fn, k=500)

  1%|          | 2/196 [00:00<00:28,  6.87it/s]

100%|██████████| 196/196 [00:27<00:00,  7.20it/s]


In [60]:
test_indices = cls30_indices_arrs[0]
train_indices = list(zip(*top500_train_cos_sim))[1]

cos_sims_arr = np.array([])
for test_idx in tqdm(test_indices):
	input, label = test_ds[test_idx]
	batch_grad_cos_fn = get_batch_grad_cos_fn(model100, input, label)

	train_top500_ds = Subset(train_ds, train_indices)
	train_top500_dl = DataLoader(train_top500_ds, 512)
	for batch in train_top500_dl:
		cos_sims = batch_grad_cos_fn(batch)

	cos_sims_arr = np.append(cos_sims_arr, cos_sims)

100%|██████████| 37/37 [00:11<00:00,  3.18it/s]


In [61]:
cos_sims_arr = np.reshape(cos_sims_arr, (len(test_indices), -1))

In [62]:
print('逐个平均cos sim')
print(np.round(np.mean(cos_sims_arr, axis=1), 2))

逐个平均cos sim
[0.4  0.27 0.37 0.19 0.34 0.32 0.26 0.3  0.32 0.23 0.29 0.31 0.34 0.29
 0.25 0.19 0.21 0.41 0.32 0.21 0.3  0.36 0.25 0.27 0.28 0.2  0.23 0.32
 0.31 0.31 0.14 0.25 0.25 0.22 0.37 0.37 0.4 ]


In [63]:
print('整体平均cos sim')
np.mean(cos_sims_arr)

整体平均cos sim


0.2872441985779851

In [64]:
test_idx = cls30_indices_arrs[0][0]
input, label = test_ds[test_idx]

# 根据该测试样本label筛选训练集对应子集
target_label = label
target_label_indices = [i for i, (_, lbl) in enumerate(train_ds) if lbl == target_label]

# 构造训练集子集和 DataLoader
train_subset = Subset(train_ds, target_label_indices)
train_dl = DataLoader(train_subset, batch_size=512, shuffle=False)

# 得到用于计算相似度的函数
batch_grad_cos_fn = get_batch_grad_cos_fn(model100, input, label)

# 收集相似度
cos_sims_list = []
for batch in tqdm(train_dl):
    batch_sim = batch_grad_cos_fn(batch)  # 返回 numpy 数组或 tensor
    # 如果是 tensor，先转 numpy
    if hasattr(batch_sim, 'cpu'):
        batch_sim = batch_sim.cpu().numpy()
    cos_sims_list.append(batch_sim)

# 拼接成一个大数组
cos_sims = np.concatenate(cos_sims_list)

# 计算均值

100%|██████████| 10/10 [00:02<00:00,  4.03it/s]


In [65]:
print("训练集样本同类别cos sim:", np.round(cos_sims.mean(), 2))

训练集样本同类别cos sim: 0.14


In [ ]:
fig, axes = plt.subplots(16, 8, figsize=(32, 64))
axes = axes.flatten()
for i in range(128):
	input = train_ds[train_indices[i]][0]
	img = inverse_trans_cifar10(input)
	img_show(img, ax=axes[i])

### using centroid to cluster

In [20]:
'''
 assume existing:
		X_pca
		hloss_samples_indices

'''

from scipy.spatial.distance import cdist
def cluster_and_get_centroids(data, n_clusters: int):
	kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
	kmeans.fit(data)

	return kmeans.labels_, kmeans.cluster_centers_


def find_closest_samples(samples, center):
	# 计算该聚类中所有样本到其中心点的距离
	distances = cdist(samples, [center], 'euclidean').flatten()
	
	# 找到最小距离及其对应的样本索引
	min_dist_idx = np.argmin(distances)
	
	return min_dist_idx


In [67]:
cls30_labels, cls30_centers = cluster_and_get_centroids(X_pca, 30)

In [77]:
len(cls30_labels[cls30_labels == 0])

37

In [78]:
test_indices = cls30_indices_arrs[0]
tar_idx = test_indices[find_closest_samples(X_pca[cls30_labels == 0], cls30_centers[0])]

In [79]:
ref_input, ref_label = test_ds[tar_idx]
fn = get_batch_grad_cos_fn(model100, ref_input, ref_label)
top500_train_center = find_topk_samples(train_ds, fn, k=500)

  0%|          | 0/196 [00:00<?, ?it/s]

100%|██████████| 196/196 [00:27<00:00,  7.19it/s]


In [80]:
def compute_test_train_cos_sims_arr(test_indices, train_indices):
	train_top500_ds = Subset(train_ds, train_indices)
	train_top500_dl = DataLoader(train_top500_ds, 512)

	cos_sims_arr = np.array([])
	for test_idx in tqdm(test_indices):
		input, label = test_ds[test_idx]
		batch_grad_cos_fn = get_batch_grad_cos_fn(model100, input, label)

		for batch in train_top500_dl:
			cos_sims = batch_grad_cos_fn(batch)

		cos_sims_arr = np.append(cos_sims_arr, cos_sims)

	cos_sims_arr = np.reshape(cos_sims_arr, (len(test_indices), -1))
	return cos_sims_arr


In [81]:
test_indices = cls30_indices_arrs[0]
train_indices = list(zip(*top500_train_center))[1]
cos_sims_arr = compute_test_train_cos_sims_arr(test_indices, train_indices)

100%|██████████| 37/37 [00:11<00:00,  3.12it/s]


In [82]:
print('用距离聚类中心最近的样本')
print('逐个平均cos sim')
print(np.round(np.mean(cos_sims_arr, axis=1), 2))
print('整体平均cos sim')
print(np.mean(cos_sims_arr))

用距离聚类中心最近的样本
逐个平均cos sim
[0.35 0.31 0.42 0.18 0.39 0.39 0.31 0.37 0.42 0.24 0.36 0.33 0.33 0.37
 0.25 0.23 0.22 0.47 0.33 0.2  0.33 0.39 0.3  0.38 0.37 0.22 0.26 0.39
 0.39 0.32 0.13 0.32 0.28 0.29 0.41 0.43 0.47]
整体平均cos sim
0.3292701895439343


### all train indices arrs

In [76]:
filtered_cls30_indices_arrs = [arr for arr in cls30_indices_arrs if len(arr) > 8]
len(filtered_cls30_indices_arrs)

24

In [87]:
train_indices_arrs = []
for i in range(30):
	test_indices = cls30_indices_arrs[i]
	if len(test_indices) > 8:
		tar_idx = test_indices[find_closest_samples(X_pca[cls30_labels == i], cls30_centers[i])]
		ref_input, ref_label = test_ds[tar_idx]
		fn = get_batch_grad_cos_fn(model100, ref_input, ref_label)
		top500_train_center = find_topk_samples(train_ds, fn, k=500)
		train_indices = list(zip(*top500_train_center))[1]
		train_indices_arrs.append(train_indices)

train_indices_arrs = np.array(train_indices_arrs) 

 29%|██▊       | 56/196 [00:08<00:18,  7.66it/s]

100%|██████████| 196/196 [00:24<00:00,  7.84it/s]


In [88]:
train_indices_arrs.shape

(24, 500)

### other baselines

In [ ]:
test_indices = cls30_indices_arrs[0]
rand_train_indices = np.random.randint(0, 50000, size=500)
cos_sims_arr = compute_test_train_cos_sims_arr(test_indices, rand_train_indices)

print('逐个平均cos sim')
print(np.round(np.mean(cos_sims_arr, axis=1), 2))
print('整体平均cos sim')
print(np.mean(cos_sims_arr))

In [ ]:
np.round(np.mean(cos_sims_arr), 2)

In [ ]:
target_label_indices = [i for i, (_, lbl) in enumerate(train_ds) if lbl == target_label]

### 10 clusters result

In [33]:
cls10_labels = dim_reduc_cluster(X_pca, 10)


使用MiniBatchKMeans进行聚类...

聚类完成！
每个聚类的样本数: [83, 40, 23, 93, 33, 39, 41, 27, 55, 66]


In [34]:
cls10_indices_arrs = \
	[hloss_samples_indices[('test', 'epoch100', 'k500')][cls10_labels == i] for i in range(10)]
cls10_labels, cls10_centers = cluster_and_get_centroids(X_pca, 10)

In [35]:
test_indices = cls10_indices_arrs[0]
test_indices[find_closest_samples(X_pca[cls10_labels == 0], cls10_centers[0])]

3607

In [36]:
train_indices_arrs = []
for i in range(10):
	test_indices = cls10_indices_arrs[i]
	tar_idx = test_indices[find_closest_samples(X_pca[cls10_labels == i], cls10_centers[i])]
	ref_input, ref_label = test_ds[tar_idx]
	fn = get_batch_grad_cos_fn(model100, ref_input, ref_label)
	top500_train_center = find_topk_samples(train_ds, fn, k=500)
	train_indices = list(zip(*top500_train_center))[1]
	train_indices_arrs.append(train_indices)

cls10_train_indices_arrs = np.array(train_indices_arrs) 

  0%|          | 0/196 [00:00<?, ?it/s]

100%|██████████| 196/196 [00:27<00:00,  7.08it/s]


## save results

In [83]:
np.save('test_top500_e100_indices.npy', hloss_samples_indices['test', 'epoch100', 'k500'])

In [90]:
np.savez_compressed('filtered_cls30_indices_arrs.npz', *filtered_cls30_indices_arrs)

In [91]:
data = np.load("filtered_cls30_indices_arrs.npz", allow_pickle=True)
filtered_cls30_indices_arrs = [data[f"arr_{i}"] for i in range(len(data.files))]

In [ ]:
np.save('train_indices_arrs.npy', train_indices_arrs)

In [37]:
np.savez_compressed('cls10_indices_arrs.npz', *cls10_indices_arrs)
np.save('cls10_train_indices_arrs.npy', cls10_train_indices_arrs)

In [32]:
train_indices_arrs[0]

array([33546, 19795, 15706, 17920, 21686, 37135,  6872, 40973, 31254,
        1815, 15210, 23306, 10527, 22244, 47915, 27250,   159, 36448,
        9570, 18584, 41421, 36414, 13093,  2565, 21982, 13458, 44029,
       12815,  1864, 24276, 36709, 37453, 24522,  4515, 46939,  1057,
       13085, 13082,  3679, 13202, 15053, 22916, 31974, 37223,  4483,
       39399,  9927, 26689, 26005,  6758, 28808, 46657,  2307, 20204,
       42999, 24518, 43385, 37513, 47075, 11632, 33525, 28554, 36474,
       21727, 18838, 49306, 24490, 49614, 17631, 45789, 48524,  6658,
        2947,   101, 37790, 26934, 36511, 35428, 26224, 11628,  7142,
       41053, 37161, 26624, 15145,  5744, 30960, 38609, 23417, 11234,
       46290, 26261, 35772, 27938, 42930, 18006, 19093, 39272, 25688,
       25677, 17431, 38085, 36795, 48052,  3902, 40086,  7492,  3762,
       16301, 16932, 12450,  4009,  8943,  7906, 23430,  5043,  6723,
       20465, 43328, 35629, 35481, 26029, 41271, 48019, 22913, 30933,
       41514, 22961,